# Match frame table

Joins every raw per-stage cache (homography, tracking, team assignment,
ball tracker, carrier assignment) into two normalized tables, once:

- **`player_frame_table`** — long format, one row per `(frame_idx, track_id)`:
  pitch position, team, role, whether that player is the ball carrier this frame,
  and that player's team's `attack_direction` for the half.
- **`ball_frame_table`** — one row per `frame_idx`: ball position, source
  (`detected`/`smoothed`/`lost`), current carrier, and `possession_team`
  (carried forward through gaps, so "who currently has the ball" stays
  defined even on frames where the carrier isn't re-confirmed).

Everything downstream (passes, pressure, possession, formation stats) reads
**only** these two tables — never the raw per-stage caches directly. That's
the whole point of this notebook: one join, done once, cached, so six stat
modules don't each re-derive a slightly different version of "where is
everyone" from six different caches.

**Note on pitch-coord validity:** the homography pipeline occasionally
produces a degenerate `H` for a frame (near-singular matrix slipping past
the degeneracy/determinant checks), which blows up every player's
projected `pitch_x`/`pitch_y` in that frame at once, far outside the
pitch's physical bounds. `build_player_frame_table` now nulls those rows
out via `flag_valid_pitch_coords` as a mandatory step (not optional, not
run ad-hoc) so garbage frames can't silently corrupt anything downstream
that aggregates over pitch coordinates. This is a symptom-level guard, not
a fix for the underlying homography bug -- that's still open in
`build_homography_cache.ipynb`.

In [8]:
import pickle
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import paths

from detector import DetectionConfig, DetectionPipeline
from detector import get_or_build_cache as get_or_build_detection_cache

from homography import HomographyConfig, HomographyEngine
from homography import get_or_build_cache as get_or_build_homography_cache

from tracker import TrackingConfig, TrackingPipeline
from tracker import get_or_build_cache as get_or_build_tracking_cache

from team_assigner import TeamAssignerConfig, TeamAssignerPipeline
from team_assigner import get_or_build_cache as get_or_build_team_cache

from ball_tracker import BallTrackerConfig, CarrierConfig, PX_PER_METER
from ball_tracker import get_or_build_ball_tracked_cache, get_or_build_ball_carrier_cache

## Load every stage's cache

Same load-or-build calls as `main.py`, minus render. Since everything is
already cached on disk, every one of these just loads a pickle — no model
inference, no recompute. Mirroring `main.py`'s exact stage order here
(rather than hand-rolling separate loading code) means this notebook can
never drift out of sync with what the actual pipeline produces.

In [9]:
# ---- 1. detection ----
det_cfg = DetectionConfig()
det_pipeline = DetectionPipeline(det_cfg)
detection_cache = get_or_build_detection_cache(
    det_pipeline, det_cfg.video_path, det_cfg.output_cache_path,
)

# ---- 2. homography ----
hom_cfg = HomographyConfig()
hom_engine = HomographyEngine(hom_cfg)
homography_cache = get_or_build_homography_cache(
    hom_engine, hom_cfg.video_path, hom_cfg.output_cache_path, ema_alpha=hom_cfg.ema_alpha,
)

# ---- 3. tracking ----
trk_cfg = TrackingConfig()
trk_pipeline = TrackingPipeline(trk_cfg)
tracking_result = get_or_build_tracking_cache(
    trk_pipeline, trk_cfg.video_path, trk_cfg.output_cache_path, detection_cache=detection_cache,
)
tracking_cache = tracking_result["tracking_cache"]
locked_class_by_id = tracking_result["locked_class_by_id"]

# ---- 4. team assignment ----
team_cfg = TeamAssignerConfig()
team_pipeline = TeamAssignerPipeline(team_cfg)
team_result = get_or_build_team_cache(
    team_pipeline, team_cfg.video_path, team_cfg.output_cache_path,
    tracking_cache=tracking_cache, locked_class_by_id=locked_class_by_id,
    homography_cache=homography_cache,
)
team_by_id = team_result["team_by_id"]

# ---- 5. ball tracking ----
ball_cfg = BallTrackerConfig()
ball_tracked_cache = get_or_build_ball_tracked_cache(
    detection_cache=detection_cache, tracking_cache=tracking_cache,
    locked_class_by_id=locked_class_by_id, homography_cache=homography_cache,
    cfg=ball_cfg, cache_path=paths.BALL_TRACKED_CACHE_PATH,
)

# ---- 6. carrier assignment ----
carrier_cfg = CarrierConfig()
ball_carrier_cache = get_or_build_ball_carrier_cache(
    ball_tracked_cache=ball_tracked_cache, tracking_cache=tracking_cache,
    locked_class_by_id=locked_class_by_id, homography_cache=homography_cache,
    cfg=carrier_cfg, cache_path=paths.BALL_CARRIER_CACHE_PATH,
)

total_frames = len(tracking_cache)
print(f"All stage caches loaded. total_frames = {total_frames}")

✅ Detection cache found at 'barca_atletico/cache/barca_atletico_detection_cache_final_v1.pkl' — loading.

Total frames cached: 3001
Ball present: 1974 (65.8%)
Ball present but low confidence: 0
Ball missing entirely: 1027 (34.2%)

Goalkeeper count/frame -> avg: 0.69, min: 0, max: 2
Referee count/frame    -> avg: 1.56, min: 0, max: 4
Player count/frame     -> avg: 21.75, min: 19, max: 25
✅ Loaded cache from 'barca_atletico/cache/homography_cache (5).pkl'.
✅ Tracking cache found at 'barca_atletico/cache/barca_atletico_tracking_cache_final_v1.pkl' — loading.
✅ Team assignment cache found at 'barca_atletico/cache/barca_atletico_team_assignment_final.pkl' — loading.
✅ Loaded cache from 'barca_atletico/cache/barca_atletico_ball_tracked_cache.pkl'.
✅ Loaded cache from 'barca_atletico/cache/barca_atletico_ball_carrier_cache.pkl'.
All stage caches loaded. total_frames = 3001


## Build the two frame tables

Pitch-coordinate conversion is batched **per frame**, not per player: every
player's foot point in a frame is projected through that frame's homography
in a single `cv2.perspectiveTransform` call. For ~20 players x 3000 frames
that's 3000 calls instead of 60,000 -- the same lesson as the annotation
render bottleneck, applied here before it becomes a problem instead of
after.

`build_player_frame_table` always applies `flag_valid_pitch_coords` before
returning -- rows from a degenerate-homography frame get `pitch_x`/`pitch_y`
nulled out, same as rows with no homography at all. This is not a separate
step to remember to run; it happens every time this function is called.

In [10]:
def bbox_foot_point(bbox):
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2.0, y2)


def flag_valid_pitch_coords(df, pitch_length, pitch_width, margin_m=5.0, verbose=True):
    """Null out pitch_x/pitch_y wherever a frame's homography produced an
    out-of-plausible-range projection (degenerate H slipping past the
    homography pipeline's own degeneracy checks, blowing up every player's
    projected position in that frame at once). This doesn't fix the
    underlying homography bug -- it stops garbage frames from silently
    corrupting anything downstream that aggregates over pitch_x/pitch_y
    (direction inference, distance calcs, press triggers, etc)."""
    df = df.copy()
    x_ok = df["pitch_x"].between(-margin_m, pitch_length + margin_m)
    y_ok = df["pitch_y"].between(-margin_m, pitch_width + margin_m)
    valid = x_ok & y_ok & df["pitch_x"].notna() & df["pitch_y"].notna()

    if verbose:
        n_bad = (~valid & df["pitch_x"].notna()).sum()
        n_total = df["pitch_x"].notna().sum()
        pct = n_bad / max(n_total, 1) * 100
        print(f"Pitch coord sanity filter -> {n_bad}/{n_total} rows ({pct:.1f}%) out of plausible range, nulled out")

    df.loc[~valid, ["pitch_x", "pitch_y"]] = np.nan
    return df


def build_player_frame_table(
    tracking_cache, locked_class_by_id, team_by_id, homography_cache,
    ball_carrier_cache, total_frames, pitch_length, pitch_width,
    px_per_meter=PX_PER_METER,
):
    """Long format: one row per (frame_idx, track_id). pitch_x/pitch_y are
    NaN wherever that frame's homography is missing (e.g. before orientation
    calibration locks in) or implausible (degenerate H) -- left as NaN
    rather than dropped, so frame counts stay comparable across columns and
    every stat module decides for itself how to handle missing pitch data."""
    rows = []
    for f in range(total_frames):
        frame_tracks = tracking_cache.get(f, {}).get("tracks", [])
        if not frame_tracks:
            continue

        H = homography_cache[f] if f < len(homography_cache) else None
        carrier_id = ball_carrier_cache.get(f, {}).get("track_id")

        feet_px = np.array([bbox_foot_point(t["bbox"]) for t in frame_tracks], dtype=np.float32)
        if H is not None:
            proj = cv2.perspectiveTransform(feet_px.reshape(-1, 1, 2), H).reshape(-1, 2)
            pitch_xy = proj / px_per_meter
        else:
            pitch_xy = np.full((len(frame_tracks), 2), np.nan)

        for t, (fx, fy), (px, py) in zip(frame_tracks, feet_px, pitch_xy):
            tid = t["track_id"]
            rows.append((
                f, tid,
                locked_class_by_id.get(tid, t["class"]),
                team_by_id.get(tid),
                float(fx), float(fy),
                float(px), float(py),
                tid == carrier_id,
            ))

    df = pd.DataFrame(rows, columns=[
        "frame_idx", "track_id", "role", "team",
        "x_px", "y_px", "pitch_x", "pitch_y", "is_carrier",
    ])
    df["team"] = df["team"].astype("Int64")
    df = flag_valid_pitch_coords(df, pitch_length, pitch_width)
    return df


def build_ball_frame_table(ball_tracked_cache, ball_carrier_cache, team_by_id, total_frames,
                            possession_gap_limit=None):
    """One row per frame_idx. possession_team forward-fills carrier_team
    through gaps (ball lost, carrier not re-confirmed yet) -- "who has the
    ball" should stay defined across a brief loose-ball frame, not reset to
    unknown every time the ball tracker or carrier assigner has a gap.

    FIX: the forward-fill is now bounded by `possession_gap_limit` (frames).
    Previously this was an unbounded ffill(), which kept attributing
    possession to the last known team indefinitely -- even after the carrier
    assigner had *deliberately* cleared carrier_track_id back to None once
    its own no_candidate_grace_frames was exceeded (see CarrierConfig).  That
    meant this table silently overrode the carrier assigner's own decision
    that possession had become genuinely unknown. Passing
    `possession_gap_limit=carrier_cfg.no_candidate_grace_frames` (the call
    sites below do this) keeps the two in sync: possession_team only bridges
    the same short gaps the carrier assigner itself considers "still held",
    and reverts to null beyond that, same as carrier_track_id does."""
    rows = []
    for f in range(total_frames):
        ball = ball_tracked_cache.get(f, {})
        carrier = ball_carrier_cache.get(f, {})
        xy_px = ball.get("xy_px")
        xy_pitch = ball.get("xy_pitch")
        carrier_id = carrier.get("track_id")
        rows.append((
            f,
            xy_px[0] if xy_px else np.nan, xy_px[1] if xy_px else np.nan,
            xy_pitch[0] if xy_pitch else np.nan, xy_pitch[1] if xy_pitch else np.nan,
            ball.get("source"), ball.get("conf"),
            carrier_id,
            team_by_id.get(carrier_id) if carrier_id is not None else None,
        ))

    df = pd.DataFrame(rows, columns=[
        "frame_idx", "ball_x_px", "ball_y_px", "ball_pitch_x", "ball_pitch_y",
        "ball_source", "ball_conf", "carrier_track_id", "carrier_team",
    ])
    df["carrier_track_id"] = df["carrier_track_id"].astype("Int64")
    df["carrier_team"] = df["carrier_team"].astype("Int64")
    df["possession_team"] = df["carrier_team"].ffill(limit=possession_gap_limit)
    return df

## Attack direction per team

Needed for anything that cares about "forward" vs "backward" (pass
direction, press triggers, progressive-pass stats, etc). Inferred from
goalkeeper position rather than team centroid drift: a keeper sits near
their own goal line almost the entire match regardless of phase of play,
which is stable even over a short continuous clip where both teams'
centroids might drift the same way during a single sustained attack (which
is exactly what happened when this was first tried with centroid drift --
it returned the same direction for both teams).

If the clip spans half-time, pass `half_boundary_frame` so direction is
computed separately per half (teams switch ends) -- otherwise leave it
`None` and direction is inferred once globally, which is correct for a
single continuous segment like this 3001-frame clip.

In [11]:
def infer_attack_direction_from_gk(player_frame_table, pitch_length, half_boundary_frame=None):
    """Infer each team's attacking direction from goalkeeper position, not
    centroid drift -- a keeper sits near their own goal line almost the
    entire match regardless of phase of play, making this stable even over
    a short continuous clip. Returns a dict keyed (team, half_idx) ->
    direction (+1 = attacks toward +x, -1 = attacks toward -x)."""
    df = player_frame_table[
        player_frame_table["pitch_x"].notna() & (player_frame_table["role"] == "goalkeeper")
    ]
    if df.empty:
        print("No goalkeeper rows found -- check the 'role' label used for keepers.")
        return {}

    total_frames = int(player_frame_table["frame_idx"].max()) + 1
    if half_boundary_frame is None:
        halves = [(0, total_frames)]
    else:
        halves = [(0, half_boundary_frame), (half_boundary_frame, total_frames)]

    direction_by_team_half = {}
    for half_idx, (start, end) in enumerate(halves):
        seg = df[(df["frame_idx"] >= start) & (df["frame_idx"] < end)]
        for team in seg["team"].dropna().unique():
            team_seg = seg[seg["team"] == team]
            gk_x = team_seg["pitch_x"].median()
            n_rows = len(team_seg)
            # keeper near x=0 -> team attacks toward +x; near x=pitch_length -> attacks toward -x
            direction_by_team_half[(int(team), half_idx)] = 1 if gk_x < pitch_length / 2 else -1
            print(f"  team {int(team)} half {half_idx}: GK median pitch_x={gk_x:.1f} (n={n_rows} rows) -> direction={direction_by_team_half[(int(team), half_idx)]}")

    return direction_by_team_half


def attach_attack_direction(player_frame_table, direction_by_team_half, half_boundary_frame=None):
    """Adds an `attack_direction` column to player_frame_table, looked up
    per row from (team, half). Rows with unknown team or no inferred
    direction get None rather than a guessed default."""
    if half_boundary_frame is None:
        half_idx = pd.Series(0, index=player_frame_table.index)
    else:
        half_idx = (player_frame_table["frame_idx"] >= half_boundary_frame).astype(int)

    player_frame_table = player_frame_table.copy()
    player_frame_table["attack_direction"] = [
        direction_by_team_half.get((int(t), h)) if pd.notna(t) else None
        for t, h in zip(player_frame_table["team"], half_idx)
    ]
    player_frame_table["attack_direction"] = player_frame_table["attack_direction"].astype("Int64")
    return player_frame_table


# Set this if the clip crosses half-time; leave None for a single continuous segment.
HALF_BOUNDARY_FRAME = None

In [12]:
player_frame_table = build_player_frame_table(
    tracking_cache, locked_class_by_id, team_by_id, homography_cache,
    ball_carrier_cache, total_frames, hom_cfg.pitch_length, hom_cfg.pitch_width,
)
ball_frame_table = build_ball_frame_table(
    ball_tracked_cache, ball_carrier_cache, team_by_id, total_frames,
    possession_gap_limit=carrier_cfg.no_candidate_grace_frames,  # FIX: bound the ffill, see build_ball_frame_table docstring
)

direction_by_team_half = infer_attack_direction_from_gk(player_frame_table, hom_cfg.pitch_length, half_boundary_frame=HALF_BOUNDARY_FRAME)
player_frame_table = attach_attack_direction(player_frame_table, direction_by_team_half, half_boundary_frame=HALF_BOUNDARY_FRAME)

print("player_frame_table:", player_frame_table.shape)
print("ball_frame_table:  ", ball_frame_table.shape)
print("attack_direction by (team, half):", direction_by_team_half)
player_frame_table.head()

Pitch coord sanity filter -> 80/70373 rows (0.1%) out of plausible range, nulled out
  team 0 half 0: GK median pitch_x=12.1 (n=1140 rows) -> direction=1
  team 1 half 0: GK median pitch_x=99.2 (n=1393 rows) -> direction=-1
player_frame_table: (70396, 10)
ball_frame_table:   (3001, 10)
attack_direction by (team, half): {(0, 0): 1, (1, 0): -1}


,frame_idx,track_id,role,team,x_px,y_px,pitch_x,pitch_y,is_carrier,attack_direction
0,0,1,referee,<NA>,1727.058838,969.111877,NaN,NaN,False,<NA>
1,0,2,referee,<NA>,672.911011,302.903595,NaN,NaN,False,<NA>
2,0,3,player,1,1031.426880,685.875366,NaN,NaN,False,-1
3,0,4,player,0,929.347046,708.467529,NaN,NaN,False,1
4,0,6,player,1,950.950745,618.303772,NaN,NaN,False,-1


## Sanity checks

In [13]:
counts = player_frame_table.groupby("frame_idx").size()
print(f"Player count/frame     -> avg: {counts.mean():.2f}, min: {counts.min()}, max: {counts.max()}")

pitch_valid = player_frame_table["pitch_x"].notna()
print(f"Pitch coord coverage   -> {pitch_valid.mean()*100:.1f}% of player-frame rows")

valid = player_frame_table[pitch_valid]
print(f"Pitch x range           -> [{valid.pitch_x.min():.1f}, {valid.pitch_x.max():.1f}]  (pitch is {hom_cfg.pitch_length}m long)")
print(f"Pitch y range           -> [{valid.pitch_y.min():.1f}, {valid.pitch_y.max():.1f}]  (pitch is {hom_cfg.pitch_width}m wide)")

ball_valid = ball_frame_table["ball_source"] != "lost"
print(f"\nBall position coverage -> {ball_valid.mean()*100:.1f}% of frames")
print(f"Carrier assigned        -> {ball_frame_table['carrier_track_id'].notna().mean()*100:.1f}% of frames")
print(f"possession_team nulls   -> {ball_frame_table['possession_team'].isna().sum()} (should be 0, or only a leading gap before the first-ever carrier)")

carrier_changes = (ball_frame_table["carrier_track_id"].fillna(-1).diff() != 0).sum()
print(f"Carrier-id changes      -> {carrier_changes} (rough upper bound on pass count -- passes.py will refine this)")

direction_valid = player_frame_table["attack_direction"].notna()
print(f"\nattack_direction coverage -> {direction_valid.mean()*100:.1f}% of player-frame rows")
print(f"attack_direction values    -> {sorted(player_frame_table['attack_direction'].dropna().unique().tolist())} (expect [-1, 1])")

if pitch_valid.mean() < 0.90 or valid.pitch_x.min() < -10 or valid.pitch_x.max() > hom_cfg.pitch_length + 10:
    print("\n\u26a0\ufe0f  Pitch coord range or coverage still looks off -- check that flag_valid_pitch_coords is actually being applied (it should print its own line above, right after the caches load).")

Player count/frame     -> avg: 23.46, min: 20, max: 24
Pitch coord coverage   -> 99.9% of player-frame rows
Pitch x range           -> [-4.6, 104.1]  (pitch is 105.0m long)
Pitch y range           -> [-5.0, 73.0]  (pitch is 68.0m wide)

Ball position coverage -> 90.0% of frames
Carrier assigned        -> 65.3% of frames
possession_team nulls   -> 807 (should be 0, or only a leading gap before the first-ever carrier)
Carrier-id changes      -> 71 (rough upper bound on pass count -- passes.py will refine this)

attack_direction coverage -> 87.8% of player-frame rows
attack_direction values    -> [-1, 1] (expect [-1, 1])


## Cache as parquet

Same load-or-build pattern as every other stage. `FORCE_REBUILD_FRAME_TABLE`
lives here (not in `paths.py`) since this notebook is currently the sole
owner of this stage's logic -- same reasoning as every other stage's
force-rebuild flag living in its own config, not in `paths.py`.

Rebuilding here always goes through `build_player_frame_table` (which
always applies `flag_valid_pitch_coords`) and `infer_attack_direction_from_gk`
-- there's no code path that produces `player_frame_table` without both of
these applied, cached or not.

In [14]:
FORCE_REBUILD_FRAME_TABLE = True


def get_or_build_frame_tables(force_rebuild=FORCE_REBUILD_FRAME_TABLE, half_boundary_frame=HALF_BOUNDARY_FRAME):
    player_path = Path(paths.PLAYER_FRAME_TABLE_CACHE_PATH)
    ball_path = Path(paths.BALL_FRAME_TABLE_CACHE_PATH)

    if player_path.exists() and ball_path.exists() and not force_rebuild:
        print(f"\u2705 Loaded frame tables from cache.")
        return pd.read_parquet(player_path), pd.read_parquet(ball_path)

    player_df = build_player_frame_table(
        tracking_cache, locked_class_by_id, team_by_id, homography_cache,
        ball_carrier_cache, total_frames, hom_cfg.pitch_length, hom_cfg.pitch_width,
    )
    ball_df = build_ball_frame_table(
        ball_tracked_cache, ball_carrier_cache, team_by_id, total_frames,
        possession_gap_limit=carrier_cfg.no_candidate_grace_frames,  # FIX: bound the ffill, see build_ball_frame_table docstring
    )

    direction_map = infer_attack_direction_from_gk(player_df, hom_cfg.pitch_length, half_boundary_frame=half_boundary_frame)
    player_df = attach_attack_direction(player_df, direction_map, half_boundary_frame=half_boundary_frame)

    player_path.parent.mkdir(parents=True, exist_ok=True)
    ball_path.parent.mkdir(parents=True, exist_ok=True)
    player_df.to_parquet(player_path, index=False)
    ball_df.to_parquet(ball_path, index=False)
    print(f"\U0001F4BE Saved frame tables to cache.")
    return player_df, ball_df


player_frame_table, ball_frame_table = get_or_build_frame_tables()

Pitch coord sanity filter -> 80/70373 rows (0.1%) out of plausible range, nulled out
  team 0 half 0: GK median pitch_x=12.1 (n=1140 rows) -> direction=1
  team 1 half 0: GK median pitch_x=99.2 (n=1393 rows) -> direction=-1
💾 Saved frame tables to cache.


## Next steps

Every stat module (`passes.py`, `pressure.py`, `possession.py`,
`formation.py`) reads `player_frame_table` / `ball_frame_table` only. None
of them touch `tracking_cache`, `homography_cache`, or any other raw
per-stage cache directly -- if pass-detection logic changes, only
`passes.py`'s own cache needs rebuilding, never this frame table.

`player_frame_table.attack_direction` is now available for any module that
needs "forward vs backward" (press triggers, progressive passes, etc.) --
no need to re-derive it downstream.

The out-of-range pitch coords seen earlier (before `flag_valid_pitch_coords`
was made a mandatory step in `build_player_frame_table`) are a symptom of
an open bug in the homography pipeline -- some frames are producing a
structurally degenerate `H`, not just an imprecise one. Worth revisiting
`build_homography_cache.ipynb`'s degeneracy/determinant checks to see why
these frames aren't being rejected there directly, rather than relying on
this notebook to catch them after the fact.